In [ ]:
import tomllib

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
from astropy import units as u
from astropy.table import Table, hstack
import astropy

In [ ]:
from astropy import constants as c

In [ ]:
from copy import deepcopy

In [ ]:
from pathlib import Path

In [ ]:
import tomli_w

In [ ]:
from scipy.special import erf

In [ ]:
import arya
arya.use("mnras")
plt.rcParams["figure.figsize"] = (8, 3)

In [ ]:
C_KMS = c.c.to("km/s").value

In [ ]:
from scipy.ndimage import gaussian_filter1d

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import sys
sys.path.append("..")

In [ ]:
import desi_utils
from desi_utils.plotting import plot_wrapped_spectrum
from desi_utils.spectral_lines import to_elsm_format
from desi_utils.fit_continuum import redshift_to_vel, vel_to_redshift
from desi_utils.catalogue_helpers import to_mel_line, to_fsf_line
from desi_utils import EMISSION_LINES, get_wavelength
from desi_utils import LineFitter
from desi_utils.fit_lines import _gaussian_flux

In [ ]:
from desiutil.dust import dust_transmission
from desispec.resolution import Resolution

In [ ]:
import h5py

In [ ]:
from desispec.io import read_spectra
from desispec.coaddition import coadd_cameras

## Data loading

In [ ]:
# dwarf_main = Table.read("../data/desi_dr1_dwarf_catalog.fits", hdu=1)
# dwarf_fsf = Table.read("../data/desi_dr1_dwarf_catalog.fits", hdu=4)

In [ ]:
# dwarf_phot = Table.read("../data/desi_dr1_dwarf_catalog.fits", hdu=3)

In [ ]:
# dwarf_cat = hstack([dwarf_main, dwarf_phot, dwarf_fsf])

In [ ]:
fsf_vac_cat1 = Table.read("../data/fastspecfit_dwarfs_part1.fits")
fsf_vac_cat2 = Table.read("../data/fastspecfit_dwarfs_part2.fits")

In [ ]:
fsf_vac_cat = hstack((fsf_vac_cat1, fsf_vac_cat2))

In [ ]:
mel_cat = Table.read("../data/stellar_mass_emline_dwarfs.fits")

In [ ]:
with open("../analysis/selected_targetid.txt", "rb") as f:
    targetids = tomllib.load(f)["targetid"]

In [ ]:
skyspec_table = Table.read("../data/skyspectrum.fits")
skyspec = desi_utils.Spectrum(wavelength=skyspec_table["wavelength"], flux=skyspec_table["flux"], uncertainty=0)

# Restart here

In [ ]:
idx_in = 0

In [ ]:
targetid = targetids[idx_in]

In [ ]:
targetid

In [ ]:
targetid in mel_cat["targetid"]

In [ ]:

np.isin(targetid, fsf_vac_cat["targetid"])

In [ ]:
out_dir = Path(".", "example_fits", str(targetid))

In [ ]:
with open(out_dir / "fluxes.toml", "rb") as f:
    mcmc_fluxes = tomllib.load(f)

In [ ]:
with open(out_dir / "ppxf_fit.toml", "rb") as f:
    ppxf_fit = tomllib.load(f)

In [ ]:
with open(out_dir / "properties.toml", "rb") as f:
    dwarf_props = tomllib.load(f)

In [ ]:
specfile = out_dir / "coadd.fits"

In [ ]:
spec_raw = read_spectra(specfile).select(targets = targetid)

In [ ]:
spec_coadd = coadd_cameras(spec_raw)

In [ ]:
band = spec_coadd.bands[0]

In [ ]:
R = Resolution(spec_coadd.resolution_data[band][0, :, :])

In [ ]:
spec_uncorr = desi_utils.Spectrum(
    redshift = dwarf_props["main"]["Z"],
    flux = spec_coadd.flux[band][0, :],
    wavelength = spec_coadd.wave[band][:],
    uncertainty = spec_coadd.ivar[band][0, :]**-0.5,

)

In [ ]:
EBV = dwarf_props["phot"]["EBV"]

In [ ]:
dust_trans = dust_transmission(spec_uncorr.wavelength, EBV)

In [ ]:
spec = spec_uncorr * (1/dust_trans)

In [ ]:
plt.plot(spec.wavelength, dust_trans)

### The ELSM fit results

# The spectrum as is

In [ ]:
specdict = {
    "observations": spec,
    # "smoothed spectrum": (spec_resamp.wavelength, convolve(spec_resamp.flux, smoothing_kernel)),
    "sky": (skyspec.wavelength, 0.1*skyspec.flux) ,
    "std": (spec.wavelength, spec.uncertainty),
}
plot_wrapped_spectrum(
    {"observations": spec},
    redshift=spec.redshift, label_lines=desi_utils.EMISSION_LINES, show_uncertainty=True,
)

plt.savefig(out_dir / "obs_spec.png")

In [ ]:
spec.wave_dispersion = np.zeros_like(spec.wavelength)

In [ ]:
smooth_spec = desi_utils.spectrum.smooth_spectrum(spec)

In [ ]:
plot_wrapped_spectrum(
    {"observations": spec,
     "sky/10": (skyspec.wavelength, skyspec.flux / 10),
    },
    {},
    redshift=spec.redshift, label_lines=desi_utils.EMISSION_LINES, show_uncertainty=True,
)

plt.savefig(out_dir / "obs_spec_w_sky.png")

In [ ]:
new_lines = [
    "H1r_3835A", #??
    "He1r_3889A", #??
    "H1r_3889A",
    "Ar4_4711A",
    "Ar4_4740A",
    "Fe2_5016A", #??
    "O1_6364A",
    # "H1r_8665A",8752, 8862, 9015, 9229
    # 6680>>>
]
    

### resampling sanity checks

In [ ]:
spec_resamp, flux_scale = desi_utils.spectrum.log_rebin_spectrum(spec)

In [ ]:
sps = desi_utils.fit_continuum.load_sps("spectra_fsps_9.0.npz", spec_resamp)

In [ ]:
np.diff(np.log10(spec_resamp.wavelength))

In [ ]:
plt.plot(np.diff(spec_resamp.wavelength) / np.diff(spec.wavelength))

In [ ]:
def plot_weight_fractions(ppxf_fit, ppxf_model, spec):
    sps = desi_utils.fit_continuum.load_sps(ppxf_model.sps_file, spec)
    light_weights = ppxf_fit.weights[~ppxf_fit.gas_component]            # Exclude weights of the gas templates
    light_weights = light_weights.reshape(ppxf_fit.reg_dim)        # Reshape to (n_ages, n_metal)

    sps.plot(light_weights/light_weights.sum())  # Normalize to light fractions


In [ ]:
plot_weight_fractions(ppxf_fit, sedmodel, spec_resamp)

In [ ]:
bestfit = np.array(ppxf_fit['bestfit'])

In [ ]:
specdict = {
    "observations": spec_resamp,
    # "smooth_spec": smooth_spec,
    "total_fit": (spec_resamp.wavelength, bestfit),
    # "my_fit": (spec_resamp.wavelength, ppxf_fit_me.bestfit),
}
styles = {
    "observations": dict(alpha=1, lw=0.3, color="k", ),
}

plot_wrapped_spectrum(specdict, styles, show_uncertainty=True, label_lines=desi_utils.EMISSION_LINES, 
                     redshift = spec.redshift)


### Spectral fit and residuals

In [ ]:
stars_bestfit = bestfit - ppxf_fit['gas_bestfit']

In [ ]:
residual = spec_resamp.flux - bestfit

In [ ]:
specdict = {
    # "smoothed spectrum": (spec_resamp.wavelength, convolve(spec_resamp.flux, smoothing_kernel)),
    "stars fit": (spec_resamp.wavelength / (1 + spec.redshift), stars_bestfit),
    "total_fit": (spec_resamp.wavelength / (1 + spec.redshift), bestfit),
    "observations": (spec_resamp.wavelength / (1 + spec.redshift), spec_resamp.flux, spec_resamp.uncertainty),
}

styles = {
    "observations": dict(alpha=1, lw=0.3, color="k", ),
}

plot_wrapped_spectrum(specdict, styles, ylim=(-0.1, np.max(spec_resamp.flux) * 1.1), 
                      label_lines=desi_utils.EMISSION_LINES,
                      show_uncertainty=True,
                     redshift=0)

plt.savefig(out_dir / "ppxf_fit_spec.png")

In [ ]:
styles = {
    "residual": dict(alpha=1, color="k"),
}

spectra = {
    "residual": (spec_resamp.wavelength, residual, spec_resamp.uncertainty),
}
plot_wrapped_spectrum(spectra, styles, ylim=(-5, 5), show_uncertainty=True, label_lines=desi_utils.EMISSION_LINES, 
                     redshift = spec.redshift)


plt.savefig(out_dir / "ppxf_residual_spec.png")

In [ ]:
fig, ax = plt.subplots()
plt.hist(1.2*residual / spec_resamp.uncertainty, bins=np.linspace(-5, 5, 100), density=True)

x = np.linspace(-5, 5, 1000)
y = 1/np.sqrt(2*np.pi) * np.exp(-x**2 / 2)
plt.plot(x, y)

ax.set(
    xlim = (-5, 5),
    xlabel = "residual $z$-score",
    ylabel = "normalized density"
)

plt.savefig(out_dir / "zscore_hist.png")

In [ ]:
styles = {
    "residual": dict(alpha=1, lw=1.0, markersize=3, marker="."),
}

spectra = {
    "residual": ((spec_resamp.wavelength), residual / spec_resamp.uncertainty),
    "zeros": ((spec_resamp.wavelength), np.zeros_like(spec_resamp.wavelength)),

}

plot_wrapped_spectrum(spectra, styles, xlabel="wavelength", ylabel="z score", ylim=(-5, 5),
label_lines=desi_utils.EMISSION_LINES,
                     redshift=spec.redshift,
                     )

plt.savefig(out_dir / "zscore_residual_spec.png")

# Finding & fitting lines

In [ ]:
flux_unit = r"$10^{-17}$ erg s$^{-1}$ cm$^{-2}$"

In [ ]:
def plot_gaussian(x_model, mu, sigma, flux, cont, dispersion=None, **kwargs):
    amp = flux / sigma / np.sqrt(2*np.pi)
    lam0 = mu #* (1 + spec.redshift)
    
    x_edges = tuple(e[:, None] for e in desi_utils.fit_lines._pixel_edges(x_model))

    y_model = _gaussian_flux(
        x_model, amp, 
        lam0, sigma,
        edges = x_edges, 
        exp = np.exp, 
        erf = erf
    )

    y_model  = dispersion @ y_model.reshape(-1)


    y_model += cont
    print(y_model)
    plt.plot(x_model, y_model.reshape(-1), **kwargs)


In [ ]:
def plot_gaussian_from_dict(x_model, flux_dict, line, dispersion=None, flux_scale=1, **kwargs):
    if line + "_flux" not in flux_dict.keys():
        print("skipping", line)
        return
    mu = flux_dict[line + "_cen"]
    sigma = flux_dict[line + "_sigma"]
    flux = flux_dict[line + "_flux"] * flux_scale
    cont = flux_dict.get(line + "_cont", 0) * flux_scale


    plot_gaussian(x_model, mu, sigma, flux, cont, dispersion=dispersion,  **kwargs)

In [ ]:
def plot_spec_subregion(spec, xlim):
    filt = spec.wavelength > xlim[0]
    filt &= spec.wavelength < xlim[1]

    plt.errorbar(spec.wavelength[filt], spec.flux[filt], yerr=spec.uncertainty[filt], fmt="ko", capsize=0)

    return spec.wavelength[filt]

In [ ]:
spec.redshift

In [ ]:
    R_filt = R.toarray()[10:20, 10:20]


In [ ]:
R_filt

In [ ]:
(np.random.normal(size=10) @ R_filt)

In [ ]:
def plot_real_fit(line):
    x0 = desi_utils.get_wavelength(line) * (1 + spec.redshift)


    xlim = x0-10, x0+10


    if xlim[0] < spec.wavelength[0]:
        return 

    fig, ax = plt.subplots(1)        

    plot_spec_subregion(spec, xlim)
    sigma_lam =  spec.wave_dispersion[np.argmin(np.abs(spec.wavelength - x0))]
    filt = (spec.wavelength > xlim[0]) & (spec.wavelength < xlim[1])
    # cont = np.median(stars_bestfit[filt])

    R_filt = R.toarray()[filt, filt]
    x_model = spec.wavelength[filt]
    plot_gaussian_from_dict(x_model, mcmc_fluxes["mcmc"], line, label="MCMC", color="C0", dispersion=R_filt)
    # plot_gaussian_from_dict(x_model, fsf_fluxes, line, label="FSF", color="C1", flux_scale = 1)
    # plot_gaussian_from_dict(x_model, fsf_vac_fluxes, line, label="FSF-VAC", color="C3", ls="--", flux_scale = 1)
    # plot_gaussian_from_dict(x_model, mel_fluxes, line, label="MEL", flux_scale=1, color="C2")
    
    ax.set(
        title = desi_utils.spectral_lines.format_line(line),
        xlabel = "observed wavelength / $\\AA$",
        ylabel = f"flux / {flux_unit}",
        ylim = 0,
        xlim = xlim
    )
    ax.legend()
    plt.savefig(out_dir / f"lines/{line}_observed.png")


In [ ]:
for line in desi_utils.EMISSION_LINES[:5]:
    plot_real_fit(line)

# Overall comparison plot

In [ ]:
plt.errorbar

In [ ]:
def plot_correlation(mc_fluxes, elsm_fluxes, ax=None, **kwargs):
    if ax is None:
        ax = plt.gca()

    xmax = 0
    xmin = 0
    
    for line in EMISSION_LINES:
        x0 = mc_fluxes.get(line + "_flux", np.nan)
        xerr = mc_fluxes.get(line + "_flux_err", np.nan)
        y0 = elsm_fluxes.get(line + "_flux", np.nan)
        yerr = elsm_fluxes.get(line + "_flux_err", np.nan)
        
        if np.isfinite(xerr) and np.isfinite(yerr) and yerr < y0:
            ms = np.log10(x0)

            ax.errorbar(x0, y0, yerr, xerr, fmt="ko", elinewidth=0.5, capsize=0, ms=ms)
            xmax = np.nanmax([x0, xmax, y0])
            xmin = np.nanmin([x0, xmin, y0])
            ax.annotate(line.replace("_", " "), (x0, y0), (5, 0),  fontsize=3, textcoords="offset points")

    ax.plot([xmin, xmax], [xmin, xmax])
    ax.set(
        aspect = 1,
        xlim=(1, xmax),
        ylim=(1, xmax),
        xscale = "log",
        yscale = "log", 
        **kwargs
    )

In [ ]:
def plot_resid(mc_fluxes, elsm_fluxes, ylabel, ax=None):
    if ax is None:
        ax = plt.gca()

    max_dev = 0
    
    for line in EMISSION_LINES:
        x0 = mc_fluxes.get(line + "_flux", np.nan)
        xerr = mc_fluxes.get(line + "_flux_err", np.nan)
        y0 = elsm_fluxes.get(line + "_flux", np.nan)
        yerr = elsm_fluxes.get(line + "_flux_err", np.nan)
        lam = mc_fluxes.get(line + "_cen", np.nan)
    
        tot_err =  (yerr/np.abs(y0) + xerr/np.abs(x0)) / np.abs(x0)
    
        if tot_err > 1:
            tot_err = 0
            color = "grey"
        else:
            color = "k"
        if np.isfinite(xerr) and np.isfinite(yerr) and (yerr < y0):
            ms = np.log10(x0)
            ax.errorbar(lam, np.log10(y0 / x0), tot_err, fmt="o", color=color, elinewidth=0.5, capsize=0, ms=ms)
            ax.annotate(line.replace("_", " "), (lam, np.log10(y0 / x0)), (5, 0),  fontsize=3, textcoords="offset points")

    
    ax.axhline(0)
    ax.set(
        xlabel = "wavelength",
        # ylim=(1 / 10**max_dev, 1 * 10**max_dev),
        ylabel = ylabel
    )

In [ ]:
def plot_both(mc_fluxes, mel_fluxes, xlabel, ylabel):
    fig, axs = plt.subplots(1, 2)

    plot_correlation(mc_fluxes, mel_fluxes, xlabel=xlabel, ylabel=ylabel, ax=axs[0])
    plot_resid(mc_fluxes, mel_fluxes, ylabel="log " + ylabel + "/" + xlabel, ax=axs[1])

In [ ]:
plot_both(mc_fluxes, mel_fluxes, xlabel="MC", ylabel="MEL")
plt.savefig(out_dir / "MC_vs_MEL_fluxes.pdf")

In [ ]:
plot_both(mc_fluxes, fsf_fluxes, xlabel="MC", ylabel="FSF-dwarf")
plt.savefig(out_dir / "MC_vs_fsf_dwarf_fluxes.pdf")

In [ ]:
plot_both(mc_fluxes, fsf_vac_fluxes, xlabel="MC", ylabel="FSF-VAC")
plt.savefig(out_dir / "MC_vs_fsf_vac_fluxes.pdf")

In [ ]:
plot_both(fsf_fluxes, fsf_vac_fluxes, xlabel="FSF-dwarf", ylabel="FSF-VAC")
plt.savefig(out_dir / "fsf_dwarf_vs_vac_fluxes.pdf")

In [ ]:
plot_both(fsf_fluxes, mel_fluxes, xlabel="FSF-dwarf", ylabel="MEL")

In [ ]:
targetid